# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [3]:
! pip install -q schedule pytest # установка библиотек, если ещё не

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [4]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [5]:
def get_book_data(book_url: str) -> dict:
    """
    Получает данные о книге по указанному URL.

    Функция выполняет GET-запрос по указанному URL, парсит HTML-страницу
    и извлекает основную информацию о книге: название, цену, рейтинг,
    количество в наличии, описание и таблицу Product Information.

    Args:
        book_url (str): Полный URL страницы книги.

    Returns:
        dict: Словарь с данными о книге. Содержит ключи:
            - 'title' (str): название книги;
            - 'price' (str): цена в фунтах (например, '£51.77');
            - 'rating' (str): рейтинг в виде слова ('One'..'Five');
            - 'availability' (str): количество в наличии
              (например, 'In stock (22 available)');
            - 'description' (str): описание книги (пустая строка, если нет);
            - 'product_information' (dict): характеристики из таблицы
              Product Information (ключ — название строки, значение — текст).
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    response = requests.get(book_url)
    response.encoding = "utf-8"

    # Проверка доступности страницы
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")

        # Название книги
        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else None

        # Цена       
        price_tag = soup.find("p", class_="price_color")
        price = price_tag.get_text(strip=True) if price_tag else None

        # Рейтинг: класс вида "star-rating Three"
        rating = None
        rating_tag = soup.find("p", class_="star-rating")
        if rating_tag:
            classes = rating_tag.get("class", [])
            for c in classes:
                if c != "star-rating":
                    rating = c
                    break

        # Наличие
        availability_tag = soup.find("p", class_="instock availability")
        availability = availability_tag.get_text(strip=True) if availability_tag else None

        # Описание: параграфы сразу после заголовка "Product Description"
        description = ''
        desc_header = soup.find("div", id="product_description")
        if desc_header:
            desc_paragraph = desc_header.find_next_sibling("p")
            if desc_paragraph:
                description = desc_paragraph.get_text(strip=True)
        
        # Таблица Product Information
        product_information = {}
        table = soup.find("table", class_="table table-striped")
        if table:
            rows = table.find_all("tr")
            for row in rows:
                th = row.find("th")
                td = row.find("td")
                if th and td:
                    key = th.get_text(strip=True)
                    value = td.get_text(strip=True)
                    product_information[key] = value                   

        # Возврат результата
        return {
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "description": description,
            "product_information": product_information,
        }

    else:
        # Возврат результата, если страница не доступна
        return f'Ошибка {response.status_code} при загрузке страницы: {book_url}'
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [6]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': '£51.77',
 'rating': 'Three',
 'availability': 'In stock (22 available)',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe 

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [7]:
import json

def scrape_books(is_save: bool = False) -> list:
    """
    Собирает ссылки на страницы всех книг из каталога.

    Проходит по страницам каталога вида
    'http://books.toscrape.com/catalogue/page-{N}.html',
    до ошибки 404 (страница не найдена).
    
    Args:
        is_save (bool): аргумент-флаг, если он будет равен `True`, то информация сохранится
                в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.
    
    Returns:
        list[str]: список книг со всеми их данными.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    books_data = []
    page_number = 1

    # Бесконечный список до ошибки 404
    while True:
        page_url = f'http://books.toscrape.com/catalogue/page-{page_number}.html'
        response = requests.get(page_url)
        response.encoding = "utf-8"

        if response.status_code == 404:
            break

        soup = BeautifulSoup(response.text, "html.parser")
        
        # На странице каталога каждая книга представлена ссылкой в <h3><a>
        book_links = soup.select("h3 a")
        if book_links:
            for link in book_links:
                href = link["href"]
                if not href:
                    continue
           
                book_url = f'http://books.toscrape.com/catalogue/{href}'
                book_info = get_book_data(book_url)
                books_data.append(book_info)
        
        page_number += 1

    # Сохранение списка с информацией о книгах в файл
    if is_save:
        with open("books_data.txt", "w", encoding="utf-8") as file:
            for book in books_data:
                file.write(json.dumps(book, ensure_ascii=False) + "\n")

    return books_data    

    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [8]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [9]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
import schedule
import time

schedule.every().day.at("19:00").do(scrape_books, is_save=True)
print("Планировщик запущен. Задача будет выполняться ежедневно в 19:00.")

while True:
    schedule.run_pending()
    time.sleep(30)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Планировщик запущен. Задача будет выполняться ежедневно в 19:00.


KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [ ]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest test/test_scraper.py

## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Решал строго по условиям заданий. Т.к. по PEP8 комментарии никогда не писал, написал как понял. Перед сдачей понял, что не исправил условие while (стоял 1 цикл) и не успел выгрузить полный каталог`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Открывал код рассматриваемой страницы и продумывал как, по каким тегам и классам лучше искать требуемое. Смотрел в конспектах и ноутбуках семинаров как мы делали то или иное действие. Во второй задаче, сначала несколько раз останавливал программу - думал, что стоп в while не срабатывает. Потом посмотрел время выполнения в первой задаче, умножил на количество книг и понял, что не дожидаюсь полной выгрузки книг.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Использовал в качестве консультанта для прояснения методов и функций, про которые не хватило информации в конспектах ассинхронных занятий и ноутбуках семинаров. Вводил промпт типа "Python schedule кратко, простыми словами", дальше читал описание и смотрел примеры. Если что-то не понимал уточнял запросами "Что делает ...?". Также использовал его для редактирования аннотаций к задачам 1 и 2`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Плохо разобрался с pytest, не совсем понял как применить написанное в конспекте к задаче`

**5. Самооценка по критериям задания**:

`Ответ: Не очень доволен собой`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Задачи понравились. Буду дальше экспериментировать с данными страницами`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

